# Zenteno · Reproducción del ajuste y varianzas exploratorias

Ajuste químico de LAB004–009 según el notebook 16 en el commit `947f97b`. Se reproducen el tratamiento de las observaciones, las iniciales, los pulsos, los pesos, la integración, las cotas y los dos arranques históricos. El MSE histórico de referencia es 3,2007190652079496.

Se estima la covarianza local por mínimos cuadrados en escala logarítmica y se convierte a unidades físicas mediante el método delta. Se muestra el cálculo con ocho coeficientes y el cálculo condicionado a fijar `Yef` en el óptimo reproducido por su cercanía a la cota superior. El segundo cálculo aporta la diagonal exploratoria de las tres tasas. Se guardan el Jacobiano completo del ajuste, los residuos ponderados y sus índices en la salida final.


In [1]:
from pathlib import Path
from dataclasses import replace
import sys,json,hashlib,math,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares,minimize_scalar
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/wine_fermentation').is_dir())
sys.path.insert(0,str(ROOT/'src'));sys.dont_write_bytecode=True
from wine_fermentation import model as kin
from wine_fermentation.data import natural_batches
from wine_fermentation.models.historical.calibration import load
from wine_fermentation.comparison import observations,chemical_observations,CHEMICAL_COLUMNS
NAME='16_zenteno_recalibration_legacy_treatment'
data,training,profiles=natural_batches(ROOT)
# Correcciones de recuento inicial indicadas por el usuario; originales conservados.
INITIAL_VIABLE_MCELLS_ML={'LAB004':10.75,'LAB005':11.25}
initial_biomass_audit=[];corrected_training=[]
for batch in training:
 if batch.batch in INITIAL_VIABLE_MCELLS_ML:
  count=INITIAL_VIABLE_MCELLS_ML[batch.batch];x0=count*.03
  ids=np.flatnonzero(np.isclose(batch.time,0.));assert len(ids)==1
  obs0={k:v.copy() for k,v in batch.observations.items()};obs0['X'][ids[0]]=x0
  initial_biomass_audit.append(dict(LAB=batch.batch,original_g_L=batch.initials['X'],viables_millones_mL=count,corregido_g_L=x0))
  batch=replace(batch,initials={**batch.initials,'X':x0},observations=obs0)
  mask=data.batch.eq(batch.batch)&np.isclose(data.time_h,0.)
  assert mask.sum()==1
  data.loc[mask,'X_viable_kg_m3']=x0;data.loc[mask,'X_viable_mcells_ml']=count
  assert batch.initials['X']==batch.observations['X'][ids[0]]
 corrected_training.append(batch)
training=corrected_training
assert len(initial_biomass_audit)==2
display(pd.DataFrame(initial_biomass_audit))

start=kin.normalize_parameters(json.loads((ROOT/'configs/theta/theta_full.json').read_text()))
config=json.loads((ROOT/'configs/models/without_transporters/calibration.json').read_text())
# Tabla 2 de Zenteno: Kn=0.01 g/L a23C; nuestro Kn0 se referencia a20C.
KN_ZENTENO_G_L=0.01;KN_PAPER_REF_K=296.15;KN_CODE_REF_K=293.15
KN_THERMAL_FACTOR=np.exp(kin.FIXED_CONSTANTS['EaKn']*(KN_PAPER_REF_K-KN_CODE_REF_K)/(KN_CODE_REF_K*kin.FIXED_CONSTANTS['R']*KN_PAPER_REF_K))
start['Kn0']=KN_ZENTENO_G_L/KN_THERMAL_FACTOR
KD0_ZENTENO_H=0.00044
start['Kd0']=KD0_ZENTENO_H
assert np.isclose(start['Kn0']*KN_THERMAL_FACTOR,KN_ZENTENO_G_L,rtol=1e-14)
display(pd.Series(dict(Kn_Zenteno_g_L=KN_ZENTENO_G_L,referencia_paper_C=23,Kn0_codigo_g_L=start['Kn0'],referencia_codigo_C=20),name='Kn fijo y conversión de referencia térmica'))
names=[k for k in config['bounds'] if k not in ['aN','KNa','Kn0','Kd0']]
assert len(names)==8 and 'Kn0' not in names and 'Kd0' not in names
bounds={k:config['bounds'][k] for k in names};lo,hi=np.array(list(bounds.values())).T
fixed={k:v for k,v in start.items() if k not in names}
display(pd.Series(fixed,name='Fijos: theta_full salvo Kn y Kd0 de Zenteno'))
display(pd.DataFrame([dict(parametro=k,inicio=start[k],inicio_factible=np.clip(start[k],lo[i]+1e-10,hi[i]-1e-10),minimo=lo[i],maximo=hi[i]) for i,k in enumerate(names)]))

,LAB,original_g_L,viables_millones_mL,corregido_g_L
0,LAB004,0.3939,10.75,0.3225
1,LAB005,0.4932,11.25,0.3375


Kn_Zenteno_g_L          0.010000
referencia_paper_C     23.000000
Kn0_codigo_g_L          0.008258
referencia_codigo_C    20.000000
Name: Kn fijo y conversión de referencia térmica, dtype: float64

Kn0     0.008258
Kg0     4.400775
Kf0    19.296084
Kd0     0.000440
Yxg     0.966094
Yxf     1.109896
m0      0.018339
Name: Fijos: theta_full salvo Kn y Kd0 de Zenteno, dtype: float64

,parametro,inicio,inicio_factible,minimo,maximo
0,mu0,0.078728,0.078728,0.05,1.0000
1,betaG0,0.428878,0.428878,0.10,10.0000
2,betaF0,3.440399,3.440399,0.10,10.0000
3,Kig0,17.845213,17.845213,5.00,1000.0000
4,Kie0,24.895476,24.895476,5.00,1000.0000
5,Yxn,3.734653,3.734653,1.00,100.0000
6,Yeg,0.250395,0.250395,0.10,0.5114
7,Yef,0.684927,0.511400,0.10,0.5114


In [2]:
diffs=[]
bylab={b.batch:b for b in training}
for a,b in [('LAB004','LAB005'),('LAB007','LAB008')]:
 aa=pd.Series(bylab[a].observations['X'],index=bylab[a].time);bb=pd.Series(bylab[b].observations['X'],index=bylab[b].time)
 z=pd.concat([aa,bb],axis=1).dropna();diffs.extend((z.iloc[:,0]-z.iloc[:,1]).tolist())
sigma_rep=float(np.sqrt(np.mean(np.square(diffs)))/np.sqrt(2));SIGMA_X=max(.5,3*sigma_rep)
assert len(diffs)>0
print('Pares de réplicas:',len(diffs),'sigma réplica:',sigma_rep,'sigmaX aplicada:',SIGMA_X)
treated=[];audit=[];rules=[]
for b in training:
 obs={k:v.copy() for k,v in b.observations.items()};old=obs['N'].copy()
 obs['N'][np.isfinite(old)&(old<=.020)]=0
 if b.batch=='LAB006':
  ids=np.flatnonzero(np.isclose(b.time,122)&np.isfinite(old));assert len(ids)==1 and np.isclose(old[ids[0]],.027)
  obs['N'][ids]=0;depletion=122.
 else:
  last=max(t for t,d in b.pulses['N']);ids=np.flatnonzero((b.time>=last)&np.isfinite(obs['N'])&(obs['N']==0));assert len(ids)
  depletion=float(b.time[ids[0]])
 obs['N'][b.time>depletion+12]=np.nan
 for i in np.flatnonzero(np.isfinite(old)):
  audit.append(dict(LAB=b.batch,t_h=b.time[i],original_mg=old[i]*1000,usado_mg=obs['N'][i]*1000,estado='Inicial' if b.time[i]==0 else ('Excluido' if not np.isfinite(obs['N'][i]) else ('Transformado' if old[i]!=obs['N'][i] else 'Usado'))))
 assert all(np.array_equal(obs[k],b.observations[k],equal_nan=True) for k in obs if k!='N')
 treated.append(replace(b,observations=obs))
 rules.append(dict(LAB=b.batch,agotamiento_h=depletion,corte_YAN_h=depletion+12,N0_mg=b.initials['N']*1000,pulsos=b.pulses['N'],**{k:int(np.count_nonzero(np.isfinite(obs[k])&(b.time>0))) for k in obs}))
audit=pd.DataFrame(audit);display(pd.DataFrame(rules));display(audit)
def sigma(state,y):return np.full_like(y,SIGMA_X) if state=='X' else kin.sigma_for_state(state,y)
def simulate(b,p):
 def rhs(t,y):
  out=np.asarray(kin.rhs(t,y,p,b))
  if y[2]<=0 or y[3]<=0:
   q=kin.kinetic_terms(p,kin.temperature_at(b,t),*y);growth=q['mu']*max(float(y[0]),0)
   out[0]-=growth;out[1]+=growth/p['Yxn'];out[2]+=growth/p['Yxg'];out[3]+=growth/p['Yxf']
  return out
 # Muestra previa al salto si coincide con la nutrición; dosis completa después.
 minimum=np.full(5,np.inf);jumps=[];events={}
 for tp,dose in b.pulses.get('N',()):
  assert tp>float(b.time[0]) and dose>=0,'No se admiten pulsos iniciales extra'
  if tp<=b.time[-1]:events[float(tp)]=events.get(float(tp),0.)+float(dose)
 times=np.asarray(b.time);state=kin.initial_vector(b);records={float(times[0]):state.copy()};now=float(times[0])
 for end in sorted(set([float(times[-1])]+list(events))):
  requested=times[(times>now)&(times<=end)]
  sol=solve_ivp(rhs,(now,end),state,t_eval=np.unique(np.r_[requested,end]),method='LSODA',rtol=2e-6,atol=2e-8,max_step=.5)
  if not sol.success or not np.isfinite(sol.y).all() or np.any(sol.y.min(axis=1)<-np.array([1e-6,1e-6,1e-4,1e-4,1e-6])):raise RuntimeError('Integración química inválida '+b.batch+' '+str(sol.message)+' mínimos='+str(dict(zip(kin.STATE_NAMES,sol.y.min(axis=1)))))
  minimum=np.minimum(minimum,sol.y.min(axis=1))
  records.update({float(t):y.copy() for t,y in zip(sol.t,sol.y.T)});state=sol.y[:,-1].copy();now=end
  if end in events:
   before=state.copy();state[1]+=events[end]
   assert np.isclose(state[1]-before[1],events[end],atol=1e-12) and np.array_equal(state[[0,2,3,4]],before[[0,2,3,4]])
   jumps.append(dict(t=end,dose=events[end],before=before.copy(),after=state.copy()))
 f=pd.DataFrame(np.maximum(np.vstack([records[float(t)] for t in times]),0),index=times,columns=kin.STATE_NAMES);f['GF']=f.G+f.F
 f.attrs['minimum_integrated_states']=dict(zip(kin.STATE_NAMES,minimum.tolist()))
 f.attrs['pulse_jumps']=jumps
 assert np.isclose(sum(j['dose'] for j in jumps),sum(events.values()),atol=1e-12)
 return f

def with_jumps(f):
 extra=[];tt=[]
 for j in f.attrs['pulse_jumps']:
  if j['t'] not in f.index:extra.append(j['before']);tt.append(j['t'])
  extra.append(j['after']);tt.append(j['t'])
 if not extra:return f
 added=pd.DataFrame(extra,index=tt,columns=kin.STATE_NAMES);added['GF']=added.G+added.F
 return pd.concat([f,added]).sort_index(kind='stable')

support=[(b,s,np.isfinite(b.observations[s])&(b.time>0)) for b in treated for s in kin.STATE_NAMES]
NR=sum(m.sum() for b,s,m in support);calls=0;rejected=0


,LAB,agotamiento_h,corte_YAN_h,N0_mg,pulsos,X,N,G,F,E
0,LAB004,91.0,103.0,230.76,"((68.5, 0.08),)",10,8,12,12,4
1,LAB005,91.0,103.0,227.30,"((68.5, 0.08),)",10,8,12,12,4
2,LAB006,122.0,134.0,229.12,"((100.0, 0.08),)",12,9,14,14,4
3,LAB007,88.0,100.0,239.04,"((65.5, 0.08),)",11,8,12,12,5
4,LAB008,88.0,100.0,239.22,"((65.5, 0.08),)",11,8,12,12,5
5,LAB009,64.0,76.0,237.40,"((41.84615384615385, 0.08),)",10,6,11,11,4


,LAB,t_h,original_mg,usado_mg,estado
0,LAB004,0.0,230.76,230.76,Inicial
1,LAB004,19.0,182.72,182.72,Usado
2,LAB004,26.0,157.42,157.42,Usado
3,LAB004,43.0,60.16,60.16,Usado
4,LAB004,50.0,19.00,0.00,Transformado
...,...,...,...,...,...
74,LAB009,88.0,17.00,NaN,Excluido
75,LAB009,96.0,19.00,NaN,Excluido
76,LAB009,120.0,5.00,NaN,Excluido
77,LAB009,144.0,25.00,NaN,Excluido


Pares de réplicas: 23 sigma réplica: 0.1189341469231937 sigmaX aplicada: 0.5


In [3]:
def residual(logp):
 global calls,rejected
 calls+=1;p={**start,**dict(zip(names,np.exp(logp)))}
 try:pred={b.batch:simulate(b,p) for b in treated}
 except (RuntimeError,ValueError,FloatingPointError):
  rejected+=1;return np.full(NR,1e4)
 e=np.concatenate([(pred[b.batch][s].to_numpy()[m]-b.observations[s][m])/sigma(s,b.observations[s][m]) for b,s,m in support])
 if calls%50==0:print('Evaluaciones',calls,'MSE',float(np.mean(e*e)),'rechazos',rejected,flush=True)
 return e
rng=np.random.default_rng(20261101);x0=np.array([start[k] for k in names]);seeds=[x0,x0*np.exp(rng.normal(0,.2,len(names)))];runs=[];runrows=[]
for j,seed in enumerate(seeds):
 sol=least_squares(residual,np.log(np.clip(seed,lo+1e-10,hi-1e-10)),bounds=(np.log(lo),np.log(hi)),max_nfev=100,diff_step=1e-3,ftol=2e-5,xtol=2e-5,gtol=2e-5)
 runs.append(sol);runrows.append(dict(inicio=j,success=sol.success,nfev=sol.nfev,MSE=float(np.mean(sol.fun**2)),message=sol.message));print(runrows[-1],flush=True)
best=min(runs,key=lambda s:s.fun@s.fun);theta={**start,**dict(zip(names,np.exp(best.x)))}
assert theta['Kd0']==KD0_ZENTENO_H and 'Kd0' not in names
assert theta['Kn0']==start['Kn0'] and np.isclose(theta['Kn0']*KN_THERMAL_FACTOR,.01,rtol=1e-14)
assert theta['Yeg']<=.5114 and theta['Yef']<=.5114 and all(theta[k]==v for k,v in fixed.items())
display(pd.DataFrame(runrows));display(pd.DataFrame([dict(parametro=k,valor=theta[k],minimo=lo[i],maximo=hi[i],cerca_cota=min(abs(np.log(theta[k]/lo[i])),abs(np.log(hi[i]/theta[k])))<.02) for i,k in enumerate(names)]))
print('PARÁMETROS QUÍMICOS COMPLETOS',json.dumps(theta,indent=2));print('Rechazos:',rejected,'Razón singular:',np.linalg.svd(best.jac,compute_uv=False)[-1]/np.linalg.svd(best.jac,compute_uv=False)[0])
traincurves={b.batch:simulate(b,theta) for b in treated}


,inicio,success,nfev,MSE,message
0,0,True,13,3.200727,`ftol` termination condition is satisfied.
1,1,True,12,3.200726,`ftol` termination condition is satisfied.


,parametro,valor,minimo,maximo,cerca_cota
0,mu0,0.094025,0.05,1.0000,False
1,betaG0,0.424387,0.10,10.0000,False
2,betaF0,0.629876,0.10,10.0000,False
3,Kig0,64.251635,5.00,1000.0000,False
4,Kie0,94.455403,5.00,1000.0000,False
5,Yxn,6.458853,1.00,100.0000,False
6,Yeg,0.468494,0.10,0.5114,False
7,Yef,0.511333,0.10,0.5114,True


Evaluaciones 50 MSE 3.2398982736328117 rechazos 0
Evaluaciones 100 MSE 3.2007284851241997 rechazos 0
{'inicio': 0, 'success': True, 'nfev': 13, 'MSE': 3.2007268154653854, 'message': '`ftol` termination condition is satisfied.'}
Evaluaciones 150 MSE 3.3508625838349566 rechazos 0
Evaluaciones 200 MSE 3.2014842956294616 rechazos 0
{'inicio': 1, 'success': True, 'nfev': 12, 'MSE': 3.2007260954991477, 'message': '`ftol` termination condition is satisfied.'}
PARÁMETROS QUÍMICOS COMPLETOS {
  "mu0": 0.09402500844265677,
  "betaG0": 0.4243869637303161,
  "betaF0": 0.6298758355545129,
  "Kn0": 0.008257860224319971,
  "Kg0": 4.400775419123265,
  "Kf0": 19.296083508118652,
  "Kig0": 64.25163532831773,
  "Kie0": 94.45540325860884,
  "Kd0": 0.00044,
  "Yxn": 6.458853242923635,
  "Yxg": 0.9660935553477866,
  "Yxf": 1.1098958529485594,
  "Yeg": 0.46849356577710505,
  "Yef": 0.5113333623796789,
  "m0": 0.0183385793928771
}
Rechazos: 0 Razón singular: 0.019943876506897374


In [4]:
# Comparación con los parámetros químicos publicados por el notebook 16 vigente.
import ast

reference_path = ROOT / 'notebooks/16_zenteno_recalibration_legacy_treatment_variable_solubility.source.ipynb'
reference_notebook = json.loads(reference_path.read_text(encoding='utf-8'))
reference_cell = next(''.join(cell['source']) for cell in reference_notebook['cells']
                      if "theta={'mu0':" in ''.join(cell['source']))
theta_reference = next(
    ast.literal_eval(node.value) for node in ast.parse(reference_cell).body
    if isinstance(node, ast.Assign)
    and any(isinstance(target, ast.Name) and target.id == 'theta' for target in node.targets)
)
mse_reference = 3.2007190652079496
mse = float(np.mean(best.fun ** 2))
assert all(run.success for run in runs)
assert rejected == 0 and len(best.fun) == 283
assert tuple(b.batch for b in treated) == tuple(f'LAB{i:03}' for i in range(4, 10))
assert abs(mse / mse_reference - 1) < 0.001
comparison = pd.DataFrame({
    'Vigente': {name: theta_reference[name] for name in names},
    'Reproducido': {name: theta[name] for name in names},
})
comparison['Diferencia relativa [%]'] = 100 * (comparison['Reproducido'] / comparison['Vigente'] - 1)
print(f'MSE ponderado histórico: {mse_reference:.12g} · reproducido: {mse:.12g}')
display(comparison)


,Vigente,Reproducido,Diferencia relativa [%]
mu0,0.094031,0.094025,-0.006876
betaG0,0.422437,0.424387,0.461654
betaF0,0.623655,0.629876,0.997440
Kig0,65.019939,64.251635,-1.181643
Kie0,96.265166,94.455403,-1.879976
Yxn,6.456226,6.458853,0.040696
Yeg,0.468459,0.468494,0.007477
Yef,0.511328,0.511333,0.001075


MSE ponderado histórico: 3.20071906521 · reproducido: 3.2007260955


In [5]:
def covariance_in_physical_units(columns):
    jacobian = np.asarray(best.jac)[:, columns]
    _, singular_values, vt = np.linalg.svd(jacobian, full_matrices=False)
    tolerance = singular_values[0] * max(jacobian.shape) * np.finfo(float).eps
    if np.any(singular_values <= tolerance):
        raise ValueError('El Jacobiano no tiene rango completo para este cálculo')
    residual_scale = float(best.fun @ best.fun) / (len(best.fun) - len(columns))
    scaled_vectors = vt.T / singular_values
    covariance_log = residual_scale * (scaled_vectors @ scaled_vectors.T)
    physical_values = np.exp(best.x[columns])
    covariance = covariance_log * np.outer(physical_values, physical_values)
    np.testing.assert_allclose(covariance, covariance.T)
    assert np.linalg.eigvalsh(covariance).min() >= -1e-12
    return covariance, singular_values, residual_scale

assert best.jac.shape == (len(best.fun), len(names))
full_columns = list(range(len(names)))
conditional_columns = [i for i, name in enumerate(names) if name != 'Yef']
conditional_names = [names[i] for i in conditional_columns]
cov_full, sv_full, scale_full = covariance_in_physical_units(full_columns)
cov_conditional, sv_conditional, scale_conditional = covariance_in_physical_units(conditional_columns)

rates = ['mu0', 'betaG0', 'betaF0']
rates_full = [names.index(name) for name in rates]
rates_conditional = [conditional_names.index(name) for name in rates]
variances = np.diag(cov_conditional)[rates_conditional]
D_theta = np.diag(variances)
display(pd.DataFrame({
    'Varianza local (8 coeficientes)': np.diag(cov_full)[rates_full],
    'Varianza local (Yef fijo)': variances,
    'Desviación estándar (Yef fijo)': np.sqrt(variances),
}, index=rates))
print('D_theta con Yef fijo, en (h^-1)^2:')
print(D_theta)
print(f'Yef = {theta["Yef"]:.9f}; cota superior = {hi[names.index("Yef")]:.7f}')


,Varianza local (8 coeficientes),Varianza local (Yef fijo),Desviación estándar (Yef fijo)
mu0,0.000010,0.000010,0.003226
betaG0,0.006015,0.001966,0.044342
betaF0,0.024077,0.015310,0.123732


D_theta con Yef fijo, en (h^-1)^2:
[[1.04076255e-05 0.00000000e+00 0.00000000e+00]
 [0.00000000e+00 1.96625390e-03 0.00000000e+00]
 [0.00000000e+00 0.00000000e+00 1.53096997e-02]]
Yef = 0.511333362; cota superior = 0.5114000


La covarianza usa la aproximación local de Gauss–Newton, con una escala común estimada de los residuos ponderados. Supone independencia y una varianza común después de ponderar. Las muestras repetidas dentro de cada fermentación pueden compartir errores, y las cotas pueden recortar la distribución. Las varianzas son exploratorias y condicionadas al modelo y los pesos; no se interpretan como incertidumbre validada entre fermentaciones. La matriz de ocho coeficientes es una comprobación de sensibilidad a incluir `Yef` en la linealización; no se usa como intervalo simétrico para ese rendimiento.

La salida JSON conserva el orden de las columnas en escala logarítmica y las etiquetas lote, variable y tiempo para cada fila. Las covarianzas y la diagonal se guardan en unidades físicas.


In [7]:
row_index = [
    {'lote': batch.batch, 'variable': state, 't_h': float(t)}
    for batch, state, mask in support for t in batch.time[mask]
]
assert len(row_index) == len(best.fun)
results = {
    'revision_historica': '947f97b',
    'mse_historico': mse_reference,
    'mse_reproducido': mse,
    'coeficientes_reproducidos': {key: float(value) for key, value in theta.items()},
    'orden_columnas_jacobiano_log': names,
    'filas': row_index,
    'residuos_ponderados': best.fun.tolist(),
    'jacobiano_residuos_log': best.jac.tolist(),
    'covarianza_local_fisica': cov_full.tolist(),
    'valores_singulares': sv_full.tolist(),
    'escala_residual': scale_full,
    'orden_covarianza_Yef_fijo': conditional_names,
    'covarianza_local_fisica_Yef_fijo': cov_conditional.tolist(),
    'valores_singulares_Yef_fijo': sv_conditional.tolist(),
    'escala_residual_Yef_fijo': scale_conditional,
    'orden_D_theta': rates,
    'D_theta_Yef_fijo': D_theta.tolist(),
}
print(json.dumps(results, allow_nan=False))


{"revision_historica": "947f97b", "mse_historico": 3.2007190652079496, "mse_reproducido": 3.2007260954991477, "coeficientes_reproducidos": {"mu0": 0.09402500844265677, "betaG0": 0.4243869637303161, "betaF0": 0.6298758355545129, "Kn0": 0.008257860224319971, "Kg0": 4.400775419123265, "Kf0": 19.296083508118652, "Kig0": 64.25163532831773, "Kie0": 94.45540325860884, "Kd0": 0.00044, "Yxn": 6.458853242923635, "Yxg": 0.9660935553477866, "Yxf": 1.1098958529485594, "Yeg": 0.46849356577710505, "Yef": 0.5113333623796789, "m0": 0.0183385793928771}, "orden_columnas_jacobiano_log": ["mu0", "betaG0", "betaF0", "Kig0", "Kie0", "Yxn", "Yeg", "Yef"], "filas": [{"lote": "LAB004", "variable": "X", "t_h": 19.0}, {"lote": "LAB004", "variable": "X", "t_h": 26.0}, {"lote": "LAB004", "variable": "X", "t_h": 43.0}, {"lote": "LAB004", "variable": "X", "t_h": 50.0}, {"lote": "LAB004", "variable": "X", "t_h": 67.0}, {"lote": "LAB004", "variable": "X", "t_h": 74.0}, {"lote": "LAB004", "variable": "X", "t_h": 91.0}, 